# Rules Discovery Using Agentic Methods

This notebook demonstrates rules extraction from policy documents using both **agentic** and **traditional** methods, with a direct comparison.

The agentic method uses Strands Agent for a self-correcting mechanism working against Pydantic models internally to ensure schema adherence.

**Important:** The agentic method only works with higher intelligence models that support tool use. Recommended model family is Anthropic Claude.

**Inputs:**
- A policy document (PDF or image)
- Discovery configuration (model, prompts, agentic toggle)

**Outputs:**
- Structured rules in `policy_classes` format (keyed on `x-aws-idp-policy-type`, ready to write into a config)
- Comparison of agentic vs traditional extraction

## 1. Setup

In [9]:
ROOTDIR = "../.."
%pip install -q -e "{ROOTDIR}/lib/idp_common_pkg[agentic-extraction]"

Note: you may need to restart the kernel to use updated packages.


In [10]:
from idp_common.extraction.agentic_idp import structured_output

In [11]:
import os, json, yaml, time, logging, boto3
from pathlib import Path
from idp_common.discovery import RulesDiscovery
from idp_common.config.models import IDPConfig

logging.basicConfig(level=logging.WARNING)
logging.getLogger('idp_common.discovery').setLevel(logging.INFO)
logging.getLogger('idp_common.bedrock').setLevel(logging.INFO)
logging.getLogger('idp_common.extraction.agentic_idp').setLevel(logging.INFO)
print("Libraries imported successfully")

Libraries imported successfully


In [12]:
POLICY_PDF_PATH = f"{ROOTDIR}/samples/rule-validation/NCCI Medicare Policy Manual.pdf"
os.environ['AWS_REGION'] = boto3.session.Session().region_name or 'us-east-1'
os.environ['METRIC_NAMESPACE'] = 'IDP-Modular-Pipeline'
region = os.environ['AWS_REGION']
print(f"AWS Region: {region}")

policy_path = Path(POLICY_PDF_PATH)
if not policy_path.exists():
    print(f"ERROR: Policy document not found at {POLICY_PDF_PATH}")
else:
    print(f"Policy document: {policy_path.name} ({policy_path.stat().st_size / (1024*1024):.1f} MB)")

AWS Region: us-east-1
Policy document: NCCI Medicare Policy Manual.pdf (0.2 MB)


In [20]:
config_dir = Path("config")
CONFIG = {}
for cf in ["rules_discovery.yaml"]:
    cp = config_dir / cf
    if cp.exists():
        with open(cp, 'r') as f:
            CONFIG.update(yaml.safe_load(f))
        print(f"Loaded {cf}")
print(f"Config sections: {list(CONFIG.keys())}")

Loaded rules_discovery.yaml
Config sections: ['discovery']


## Agentic Rules Discovery
### 2. Configure - with Agentic

In [ ]:
# Enable agentic mode — preserve other agentic settings (review_agent, review_agent_model) from YAML
CONFIG["discovery"]["rules"].setdefault("agentic", {})["enabled"] = True
CONFIG["discovery"]["rules"]["model"] = "us.anthropic.claude-opus-4-6-v1"
# CONFIG["discovery"]["rules"]["model"] = "us.anthropic.claude-opus-4-6-v1:1m"

rc = CONFIG['discovery']['rules']
print(f"Model: {rc['model']}")
print(f"Agentic: {rc['agentic']}")
print(f"Temperature: {rc.get('temperature')}")
print(f"Max Tokens: {rc.get('max_tokens')}")

Model: us.anthropic.claude-opus-4-6
Agentic: {'enabled': True, 'review_agent': True, 'review_agent_model': 'us.anthropic.claude-sonnet-4-6'}
Temperature: 0.0
Max Tokens: 64000


### 3. Run Agentic Rules Discovery

In [ ]:
config_agentic = IDPConfig(**CONFIG)
discovery_agentic = RulesDiscovery(input_bucket="unused", input_prefix="unused", region=region, config=config_agentic)

print(f"Model: {discovery_agentic.rules_config.model}")
print(f"Agentic: {discovery_agentic.rules_config.agentic.enabled}")
print("Extracting rules...\n")

t0 = time.time()
result_agentic = discovery_agentic.discovery_rules_from_document_local(POLICY_PDF_PATH)
agentic_time = time.time() - t0

print(f"\nCompleted in {agentic_time:.1f}s | Status: {result_agentic['status']} | Rule classes: {len(result_agentic['rules'])}")

### 4. Display Agentic Results

In [ ]:
rules_agentic = result_agentic['rules']
total_agentic = 0
for i, rc in enumerate(rules_agentic):
    rp = rc.get('rule_properties', {})
    total_agentic += len(rp)
    print(f"\n[{rc.get('x-aws-idp-policy-type') or rc.get('x-aws-idp-rule-type','?')}] {len(rp)} rules")
    for rid, rdef in list(rp.items())[:3]:
        d = rdef.get('description','')[:100]
        print(f"  {rid} (p.{rdef.get('page','?')}): {d}")
    if len(rp) > 3:
        print(f"  ... +{len(rp)-3} more")
print(f"\nTOTAL: {len(rules_agentic)} classes, {total_agentic} rules")

### 5. Save Agentic Results

In [ ]:
data_dir = Path(".data/step_rules_discovery_agentic")
data_dir.mkdir(parents=True, exist_ok=True)
with open(data_dir / "policy_classes_agentic.json", 'w') as f:
    json.dump(rules_agentic, f, indent=2)
print(f"Saved agentic rules to {data_dir}")

---
## Traditional Rules Discovery
### 2. Configure - without Agentic

In [ ]:
# Disable agentic mode for traditional comparison run
CONFIG["discovery"]["rules"].setdefault("agentic", {})["enabled"] = False
CONFIG["discovery"]["rules"]["model"] = "us.anthropic.claude-sonnet-5"

rc = CONFIG['discovery']['rules']
print(f"Model: {rc['model']}")
print(f"Agentic: {rc['agentic']}")

### 3. Run Traditional Rules Discovery

In [ ]:
config_traditional = IDPConfig(**CONFIG)
discovery_traditional = RulesDiscovery(input_bucket="unused", input_prefix="unused", region=region, config=config_traditional)

print(f"Model: {discovery_traditional.rules_config.model}")
print(f"Agentic: {discovery_traditional.rules_config.agentic.enabled}")
print("Extracting rules...\n")

t0 = time.time()
result_traditional = discovery_traditional.discovery_rules_from_document_local(POLICY_PDF_PATH)
traditional_time = time.time() - t0

print(f"\nCompleted in {traditional_time:.1f}s | Status: {result_traditional['status']} | Rule classes: {len(result_traditional['rules'])}")

### 4. Display Traditional Results

In [ ]:
rules_traditional = result_traditional['rules']
total_traditional = 0
for i, rc in enumerate(rules_traditional):
    rp = rc.get('rule_properties', {})
    total_traditional += len(rp)
    print(f"\n[{rc.get('x-aws-idp-policy-type') or rc.get('x-aws-idp-rule-type','?')}] {len(rp)} rules")
    for rid, rdef in list(rp.items())[:3]:
        d = rdef.get('description','')[:100]
        print(f"  {rid} (p.{rdef.get('page','?')}): {d}")
    if len(rp) > 3:
        print(f"  ... +{len(rp)-3} more")
print(f"\nTOTAL: {len(rules_traditional)} classes, {total_traditional} rules")

### 5. Save Traditional Results

In [ ]:
with open(data_dir / "policy_classes_traditional.json", 'w') as f:
    json.dump(rules_traditional, f, indent=2)
print(f"Saved traditional rules to {data_dir}")

---
## Direct Comparison: Same Document, Both Methods

In [ ]:
def count_rules(rules_list):
    return sum(len(rc.get('rule_properties', {})) for rc in rules_list)

def page_ref_stats(rules_list):
    total, with_page = 0, 0
    for rc in rules_list:
        for rd in rc.get('rule_properties', {}).values():
            total += 1
            if rd.get('page', '').strip(): with_page += 1
    return with_page, total

def key_stats(rules_list):
    generic, descriptive = 0, 0
    for rc in rules_list:
        for k in rc.get('rule_properties', {}).keys():
            if k.startswith('rule_') and k[5:].isdigit(): generic += 1
            else: descriptive += 1
    return descriptive, generic

a_count = count_rules(rules_agentic)
t_count = count_rules(rules_traditional)
a_pg, a_pg_tot = page_ref_stats(rules_agentic)
t_pg, t_pg_tot = page_ref_stats(rules_traditional)
a_desc, a_gen = key_stats(rules_agentic)
t_desc, t_gen = key_stats(rules_traditional)

print(f"Comparing: {Path(POLICY_PDF_PATH).name}")
print("=" * 60)

print(f"\n  TRADITIONAL")
print(f"  Time: {traditional_time:.1f}s | Classes: {len(rules_traditional)} | Rules: {t_count}")
print(f"  Page refs: {t_pg}/{t_pg_tot} | Keys: {t_desc} descriptive, {t_gen} generic")

print(f"\n  AGENTIC")
print(f"  Time: {agentic_time:.1f}s | Classes: {len(rules_agentic)} | Rules: {a_count}")
print(f"  Page refs: {a_pg}/{a_pg_tot} | Keys: {a_desc} descriptive, {a_gen} generic")

print("\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)

if traditional_time > 0:
    spd = ((traditional_time - agentic_time) / traditional_time) * 100
    print(f"Speed: Agentic is {spd:.1f}% {'faster' if spd > 0 else 'slower'}")
    print(f"  Traditional: {traditional_time:.1f}s | Agentic: {agentic_time:.1f}s")

rd = a_count - t_count
print(f"\nRules: Agentic extracted {rd:+d} rules")
print(f"  Traditional: {t_count} | Agentic: {a_count}")

print(f"\nSchema Compliance:")
print(f"  Traditional: Manual JSON parsing + validation retries")
print(f"  Agentic: Pydantic model enforced via tool-based validation (guaranteed schema)")

print(f"\nKey Quality:")
print(f"  Traditional: {t_desc} descriptive, {t_gen} generic")
print(f"  Agentic: {a_desc} descriptive, {a_gen} generic")

print("\nKEY ADVANTAGES OF AGENTIC RULE DISCOVERY:")
advs = []
if rd > 0: advs.append(f"Better rule coverage (+{rd} rules)")
if a_gen == 0 and t_gen > 0: advs.append("All descriptive rule names (no generic rule_N)")
if a_pg > t_pg: advs.append(f"Better page references ({a_pg} vs {t_pg})")
advs.append("Self-correcting with Pydantic schema validation")
advs.append("Handles large docs via buffer/patch tools")
advs.append("Guaranteed output format, no JSON parsing failures")
for a in advs: print(f"  - {a}")

In [ ]:
# Side-by-side: first 3 rules from each method
def sample_rules(rules_list, n=3):
    out = []
    for rc in rules_list:
        for rid, rd in rc.get('rule_properties', {}).items():
            out.append((rid, rd))
            if len(out) >= n: return out
    return out

print("TRADITIONAL (first 3):")
for rid, rd in sample_rules(rules_traditional, 3):
    print(f"  [{rid}] (p.{rd.get('page','?')}) {rd.get('description','')[:100]}")

print("\nAGENTIC (first 3):")
for rid, rd in sample_rules(rules_agentic, 3):
    print(f"  [{rid}] (p.{rd.get('page','?')}) {rd.get('description','')[:100]}")

## Summary

In [ ]:
print("=== Rules Discovery Agentic Comparison Complete ===")
print(f"Document: {Path(POLICY_PDF_PATH).name}")
print(f"Agentic: {len(rules_agentic)} classes, {count_rules(rules_agentic)} rules in {agentic_time:.1f}s")
print(f"Traditional: {len(rules_traditional)} classes, {count_rules(rules_traditional)} rules in {traditional_time:.1f}s")
print(f"Data saved to: .data/step_rules_discovery_agentic/")
print("\nNext: Use these rules with RuleValidationService to validate transactions")